In [7]:
# ==============================================================================
# Raw Data Collection, Chronological Splitting, and Athena Datalake
# ==============================================================================
# Description: 
# This notebook fetches raw quantitative market data, splits it chronologically 
# (Train 40%, Val 10%, Test 10%, Prod 40%) to prevent lookahead bias, and uploads 
# the partitions to an S3 Datalake. Finally, it provisions AWS Athena tables 
# for scalable querying prior to Feature Engineering.

!pip install -q yfinance
import yfinance as yf
import pandas as pd
import boto3
import time
import io
import sagemaker
from sagemaker.core.helper.session_helper import Session

# --- AWS Configuration ---
# Initializing SageMaker session to automatically grab the default bucket
session = Session()
S3_BUCKET = session.default_bucket() 
PREFIX_QUANT = "quantamental-platform/raw-data/quantitative"
PREFIX_NEWS = "quantamental-platform/raw-data/qualitative"
ATHENA_DB = "quantamental_risk_db"

print(f"Using S3 Bucket: {S3_BUCKET}")

Using S3 Bucket: sagemaker-us-east-1-025891565789


### Fetching and Splitting the Data

In [8]:
TICKERS = ["VOO", "QQQ", "SPY", "IWM", "DIA", "GLD", "^VIX"] 
START_DATE = "2014-01-01"
END_DATE = "2026-01-01"

print("Fetching historical market data...")
df_list = []
for ticker in TICKERS:
    data = yf.download(ticker, start=START_DATE, end=END_DATE)
    if isinstance(data.columns, pd.MultiIndex):
        data.columns = [col[0] for col in data.columns]
    data = data.reset_index()
    data.rename(columns={data.columns[0]: 'Date'}, inplace=True)
    data['Ticker'] = ticker
    df_list.append(data)

raw_df = pd.concat(df_list)

# THE CRITICAL FIX: Strip timezones and lock exact columns
# Athena chokes on timezone strings and extra columns (like 'Adj Close')
raw_df['Date'] = pd.to_datetime(raw_df['Date'], utc=True).dt.tz_localize(None)

# Force a strict 7-column schema
required_cols = ['Date', 'Open', 'High', 'Low', 'Close', 'Volume', 'Ticker']
raw_df = raw_df[required_cols].copy()
raw_df.sort_values(by=['Date', 'Ticker'], inplace=True)

# --- 2. Chronological Data Split ---
total_rows = len(raw_df)
train_idx = int(total_rows * 0.40)
val_idx = train_idx + int(total_rows * 0.10)
test_idx = val_idx + int(total_rows * 0.10)

train_df = raw_df.iloc[:train_idx]
val_df = raw_df.iloc[train_idx:val_idx]
test_df = raw_df.iloc[val_idx:test_idx]
prod_df = raw_df.iloc[test_idx:]

print(f"Data Split Complete. Total records: {total_rows}")

[*********************100%***********************]  1 of 1 completed

Fetching historical market data...



[*********************100%***********************]  1 of 1 completed

[*********************100%***********************]  1 of 1 completed

[*********************100%***********************]  1 of 1 completed

[*********************100%***********************]  1 of 1 completed

[*********************100%***********************]  1 of 1 completed

[*********************100%***********************]  1 of 1 completed

Data Split Complete. Total records: 21126


 ### Uploading to S3 Datalake

In [9]:
s3_client = boto3.client('s3')

def upload_df_to_s3(df, split_name):
    csv_buffer = io.StringIO()
    # date_format ensures Athena's strict timestamp parser can read it
    df.to_csv(csv_buffer, index=False, date_format='%Y-%m-%d %H:%M:%S')
    s3_key = f"{PREFIX_QUANT}/{split_name}/market_data.csv"
    s3_client.put_object(Bucket=S3_BUCKET, Key=s3_key, Body=csv_buffer.getvalue())

print("Uploading clean data to S3...")
upload_df_to_s3(train_df, "train")
upload_df_to_s3(val_df, "validation")
upload_df_to_s3(test_df, "test")
upload_df_to_s3(prod_df, "production")

Uploading clean data to S3...


### Setting up Amazon Athena

In [10]:
# --- 4. Set up Athena Database and Tables ---
athena_client = boto3.client('athena')

def run_athena_query(query, database=None):
    kwargs = {'QueryString': query, 'ResultConfiguration': {'OutputLocation': f"s3://{S3_BUCKET}/athena-results/"}}
    if database:
        kwargs['QueryExecutionContext'] = {'Database': database}
    response = athena_client.start_query_execution(**kwargs)
    execution_id = response['QueryExecutionId']
    while True:
        response = athena_client.get_query_execution(QueryExecutionId=execution_id)
        state = response['QueryExecution']['Status']['State']
        if state in ['SUCCEEDED', 'FAILED', 'CANCELLED']:
            break
        time.sleep(2)
    if state != 'SUCCEEDED':
        raise Exception(f"Athena query failed: {response['QueryExecution']['Status'].get('StateChangeReason')}")

print("Rebuilding Athena Schema...")
run_athena_query(f"CREATE DATABASE IF NOT EXISTS {ATHENA_DB}")
run_athena_query(f"DROP TABLE IF EXISTS {ATHENA_DB}.market_data_train")

# Notice exactly 7 columns to match our Python dataframe
ddl_query = f"""
CREATE EXTERNAL TABLE IF NOT EXISTS {ATHENA_DB}.market_data_train (
    `Date` timestamp,
    `Open` double,
    `High` double,
    `Low` double,
    `Close` double,
    `Volume` bigint,
    `Ticker` string
)
ROW FORMAT DELIMITED
FIELDS TERMINATED BY ','
STORED AS TEXTFILE
LOCATION 's3://{S3_BUCKET}/{PREFIX_QUANT}/train/'
TBLPROPERTIES ('skip.header.line.count'='1');
"""
run_athena_query(ddl_query, database=ATHENA_DB)
print("Notebook 1 Complete: Data is strictly typed, formatted, and safely queryable!")

Rebuilding Athena Schema...


Notebook 1 Complete: Data is strictly typed, formatted, and safely queryable!
